# 05 — Advanced Models (Week 7)

**Gradient boosting** vs. the Week 4–6 models. It uses scikit-learn's `HistGradientBoostingRegressor`, a histogram-based booster from the same algorithm family as LightGBM. LightGBM and XGBoost both need a system OpenMP library on macOS, and this one runs anywhere scikit-learn does.

Protocol (same as Week 5, best-practices §09):
1. Tune **depth, learning rate and number of trees** on the **validation month** (trained on the 18 months before it). The grid has 12 settings.
2. Score the **test month once** with the chosen settings.
3. Re-train on all six Week 6 feature sets, then re-run selection on validation MdAPE with the 0.10-pt noise rule.

`scripts/run_advanced_models.py` runs the same steps and saves the selected model.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idx_avm import config
from idx_avm.pipeline import prepare_dataset
from idx_avm.features import FEATURE_SETS
from idx_avm.models import MODEL_GRIDS, fit_and_score, tune_on_validation, select_model, permutation_mdape_importance, NOISE_TOLERANCE_PTS

X_MONTHS = json.loads((config.PROCESSED_DIR / "split_metadata.json").read_text())["chosen_window_months"]
data = prepare_dataset(week6_features=True)
DF, TEST_MONTH = data.df, data.months[-1]
VAL_MONTH = TEST_MONTH - 1
GB = "Gradient Boosting"
print(f"{len(DF):,} rows | train window {X_MONTHS} months | validation {VAL_MONTH} | test {TEST_MONTH}")
print(f"{len(MODEL_GRIDS[GB][1])} gradient boosting settings to try")

## 1. Light hyperparameter tuning (validation month only)

In [ ]:
gb_params, trials = tune_on_validation(DF, TEST_MONTH, X_MONTHS, GB, flags={})
trials.sort_values("MdAPE")[["params", "MdAPE", "R2", "fit_seconds"]].round(3)

In [ ]:
# How each knob moves validation error
t = trials.copy()
for k in ("max_depth", "learning_rate", "max_iter"):
    t[k] = t["params"].str.extract(rf"{k}=([^,]+)")[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, k in zip(axes, ("max_depth", "learning_rate", "max_iter")):
    t.groupby(k)["MdAPE"].mean().plot.bar(ax=ax, color="#ea580c", title=f"Mean val MdAPE by {k}")
    ax.set_ylabel("MdAPE (%)")
plt.tight_layout(); plt.show()
print("Chosen:", gb_params)

## 2. Test-month metrics, side by side with the baseline (base features)

In [ ]:
week5 = pd.read_csv(config.PROCESSED_DIR / "week5_model_comparison.csv")
gb_base = fit_and_score(DF, TEST_MONTH, X_MONTHS, GB, gb_params, flags={}, feature_set="Base (Week 4)")
table = pd.concat([week5[["model", "MdAPE", "MAPE", "R2", "MAE", "RMSE"]],
                   pd.DataFrame([{"model": GB, **{k: gb_base.metrics[k] for k in ("MdAPE", "MAPE", "R2", "MAE", "RMSE")}}])])
base = table.loc[table.model == "Linear Regression", "MdAPE"].iloc[0]
table["MdAPE vs baseline (%)"] = (table["MdAPE"] - base) / base * 100
table.round(3)

## 3. Gradient boosting on every Week 6 feature set

In [ ]:
rows, fitted = [], {}
for set_name, flags in FEATURE_SETS.items():
    val = fit_and_score(DF, VAL_MONTH, X_MONTHS, GB, gb_params, flags, set_name)
    test = fit_and_score(DF, TEST_MONTH, X_MONTHS, GB, gb_params, flags, set_name)
    fitted[set_name] = test
    rows.append({"feature_set": set_name, "model": GB, "val_MdAPE": val.metrics["MdAPE"],
                 **{f"test_{k}": v for k, v in test.metrics.items() if k != "n"}})
gb_sets = pd.DataFrame(rows)
gb_sets.round(3)

In [ ]:
all_results = pd.concat([pd.read_csv(config.PROCESSED_DIR / "week6_feature_sets.csv"), gb_sets], ignore_index=True)
ax = None
for model, g in all_results[all_results.model.isin(["Linear Regression", "Random Forest", GB])].groupby("model"):
    ax = g.set_index("feature_set").loc[list(FEATURE_SETS), "test_MdAPE"].plot(marker="o", label=model, ax=ax, figsize=(10, 4))
ax.set_ylabel("Test MdAPE (%)"); ax.legend(); plt.xticks(rotation=20); plt.show()

## 4. Selection across Weeks 5–7 (validation only, same noise rule)

In [ ]:
lowest = all_results.loc[all_results.val_MdAPE.idxmin()]
best = select_model(all_results, list(FEATURE_SETS))
print(f"Lowest validation MdAPE: {lowest.model} with {lowest.feature_set} ({lowest.val_MdAPE:.2f}%)")
print(f"Selected (simplest within {NOISE_TOLERANCE_PTS} pts): {best.model} with {best.feature_set} ({best.val_MdAPE:.2f}%)")

In [ ]:
sel = fitted[best.feature_set] if best.model == GB else None
if sel is not None:
    imp = permutation_mdape_importance(sel)
    imp.head(12).plot.barh(x="feature", y="mdape_increase_pts", xerr="std", legend=False, figsize=(7, 5), color="#ea580c").invert_yaxis()
    plt.xlabel("Rise in test MdAPE when shuffled (pts)"); plt.title("Permutation importance, selected model"); plt.show()

## 5. Takeaways (fill in from the run)

- Test MdAPE, R² and MAE of gradient boosting vs. the Random Forest and the baseline, from the same table.
- Which settings won, and what the depth / learning-rate / trees chart shows.
- Whether boosting uses the Week 6 comps features better than the Random Forest did.
- Saved model size compared with the Random Forest (~350 MB).